In [0]:
# BLOQUE 1 — Parámetros del snapshot y resolución controlada de la fuente

import re

from pyspark.sql import functions as F

catalog = "workspace"
schema = "gtfs_bogota"
volume = "raw_gtfs"
raw_volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
control_table = f"{catalog}.{schema}.control_gtfs_static_catalog"

# El parámetro identifica la versión del proveedor. La URL no se recibe como
# texto libre: se resuelve desde el catálogo descubierto y auditado.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
feed_version = dbutils.widgets.get("feed_version").strip()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not spark.catalog.tableExists(control_table):
    raise RuntimeError(
        f"No existe {control_table}. Ejecuta primero el notebook 06."
    )

# Un ítem directo y publicado es el único candidato válido para Raw.
source_candidates_df = (
    spark.table(control_table)
    .filter(
        (F.col("feed_version") == feed_version)
        & F.col("is_direct_zip_url")
    )
    .orderBy(F.col("portal_modified_at_utc").desc())
    .limit(1)
)

source_candidate = source_candidates_df.first()
if source_candidate is None:
    raise RuntimeError(
        f"No existe un ZIP directo controlado para la versión {feed_version}."
    )

current_feed = {
    "feed_version": feed_version,
    "source_url": source_candidate["source_url"],
    "portal_item_id": source_candidate["portal_item_id"],
}

spark.sql(f"use catalog `{catalog}`")
spark.sql(f"use schema `{schema}`")

display(source_candidates_df)

In [0]:
# BLOQUE 2 — Descarga de la versión GTFS seleccionada

# La URL fue resuelta desde el catálogo de control para la versión solicitada.
# La descarga produce evidencia técnica: run_id, UTC, tamaño, HTTP y hash.

from datetime import datetime, timezone
import hashlib
import uuid

import requests

run_id = str(uuid.uuid4())
captured_at_utc = datetime.now(timezone.utc)

response = requests.get(
    current_feed["source_url"],
    headers={
        "User-Agent": "bogota-gtfs-mobility-lakehouse/0.1"
    },
    timeout=(10, 120),
)

# Si la fuente no responde 200, el notebook se detiene aquí.
response.raise_for_status()

zip_bytes = response.content

download_metadata = {
    "run_id": run_id,
    "captured_at_utc": captured_at_utc.isoformat(),
    "feed_version": current_feed["feed_version"],
    "source_url": current_feed["source_url"],
    "http_status": response.status_code,
    "bytes_received": len(zip_bytes),
    "sha256": hashlib.sha256(zip_bytes).hexdigest(),
}

display(spark.createDataFrame([download_metadata]))

In [0]:
# BLOQUE 3 — Validación estructural del ZIP GTFS
#verificamos que era un ZIP real y que contiene los archivos GTFS esenciales:
#agency, stops, routes, trips, stop_times y calendario.
import io
import json
import zipfile

from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

zip_is_valid = zipfile.is_zipfile(io.BytesIO(zip_bytes))

if not zip_is_valid:
    raise RuntimeError("La descarga no corresponde a un archivo ZIP válido.")

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as gtfs_zip:
    zip_files = sorted(
        file_name
        for file_name in gtfs_zip.namelist()
        if not file_name.endswith("/")
    )

file_names = {
    file_name.rsplit("/", 1)[-1]
    for file_name in zip_files
}

required_files = {
    "agency.txt",
    "stops.txt",
    "routes.txt",
    "trips.txt",
    "stop_times.txt",
}

missing_required_files = sorted(required_files - file_names)

has_calendar_definition = (
    "calendar.txt" in file_names
    or "calendar_dates.txt" in file_names
)

validation_passed = (
    len(missing_required_files) == 0
    and has_calendar_definition
)

validation_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("feed_version", StringType(), False),
    StructField("http_status", IntegerType(), False),
    StructField("bytes_received", LongType(), False),
    StructField("sha256", StringType(), False),
    StructField("zip_is_valid", BooleanType(), False),
    StructField("has_calendar_definition", BooleanType(), False),
    StructField("validation_passed", BooleanType(), False),
    StructField("missing_required_files_json", StringType(), False),
    StructField("zip_files_json", StringType(), False),
])

validation_row = [(
    run_id,
    current_feed["feed_version"],
    response.status_code,
    len(zip_bytes),
    download_metadata["sha256"],
    zip_is_valid,
    has_calendar_definition,
    validation_passed,
    json.dumps(missing_required_files),
    json.dumps(zip_files),
)]

validation_df = spark.createDataFrame(
    validation_row,
    schema=validation_schema,
)

display(validation_df)

In [0]:
# BLOQUE 4 — Persistencia idempotente del GTFS estático en Raw

import json

from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

feed_name = "gtfs_static"
archive_file_name = "gtfs.zip"

snapshot_dir = (
    f"{raw_volume_path}/{feed_name}/"
    f"feed_version={current_feed['feed_version']}/"
    f"sha256={download_metadata['sha256']}"
)

raw_snapshot_path = f"{snapshot_dir}/{archive_file_name}"

ingest_date = captured_at_utc.strftime("%Y-%m-%d")

manifest_dir = (
    f"{raw_volume_path}/{feed_name}/manifests/"
    f"ingest_date={ingest_date}"
)

manifest_file_name = f"run_id={run_id}.json"
manifest_path = f"{manifest_dir}/{manifest_file_name}"

dbutils.fs.mkdirs(snapshot_dir)
dbutils.fs.mkdirs(manifest_dir)

existing_files = [
    file.name
    for file in dbutils.fs.ls(snapshot_dir)
]

snapshot_already_exists = archive_file_name in existing_files

# El ZIP se identifica por su hash. Si existe, no se sobrescribe evidencia Raw.
if not snapshot_already_exists:
    with open(raw_snapshot_path, "wb") as file:
        file.write(zip_bytes)

new_manifest = {
    "run_id": run_id,
    "captured_at_utc": captured_at_utc.isoformat(),
    "feed_name": feed_name,
    "feed_version": current_feed["feed_version"],
    "source_url": current_feed["source_url"],
    "http_status": response.status_code,
    "bytes_received": len(zip_bytes),
    "sha256": download_metadata["sha256"],
    "raw_snapshot_path": raw_snapshot_path,
    "snapshot_already_exists": snapshot_already_exists,
    "zip_is_valid": zip_is_valid,
    "has_calendar_definition": has_calendar_definition,
    "validation_passed": validation_passed,
    "missing_required_files": missing_required_files,
}

# Reejecutar este bloque no debe fallar por reutilizar el mismo run_id.
# Si ya existe el manifiesto, verificamos que represente exactamente el mismo feed.
existing_manifest_files = {
    file.name
    for file in dbutils.fs.ls(manifest_dir)
}
manifest_already_exists = manifest_file_name in existing_manifest_files

if manifest_already_exists:
    manifest = json.loads(dbutils.fs.head(manifest_path))
    manifest_matches_run = (
        manifest.get("feed_version") == current_feed["feed_version"]
        and manifest.get("sha256") == download_metadata["sha256"]
        and manifest.get("raw_snapshot_path") == raw_snapshot_path
    )
    if not manifest_matches_run:
        raise RuntimeError(
            "El run_id existente apunta a una evidencia Raw diferente. "
            "No se sobrescribirá el manifiesto."
        )
else:
    manifest = new_manifest
    dbutils.fs.put(
        manifest_path,
        json.dumps(manifest, indent=2),
        overwrite=False,
    )

# Un esquema explícito permite mostrar listas vacías sin depender de inferencia.
manifest_display_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("captured_at_utc", StringType(), False),
    StructField("feed_name", StringType(), False),
    StructField("feed_version", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("http_status", IntegerType(), False),
    StructField("bytes_received", LongType(), False),
    StructField("sha256", StringType(), False),
    StructField("raw_snapshot_path", StringType(), False),
    StructField("snapshot_already_exists", BooleanType(), False),
    StructField("zip_is_valid", BooleanType(), False),
    StructField("has_calendar_definition", BooleanType(), False),
    StructField("validation_passed", BooleanType(), False),
    StructField("missing_required_files", ArrayType(StringType()), False),
    StructField("manifest_already_exists", BooleanType(), False),
])

manifest_display = {
    **manifest,
    "manifest_already_exists": manifest_already_exists,
}
display(
    spark.createDataFrame(
        [manifest_display],
        schema=manifest_display_schema,
    )
)

# En un Lakeflow Job estas salidas conectan Raw con Staging y Bronze.
# Fuera de un Job el notebook conserva su comportamiento interactivo.
task_values = {
    "feed_version": current_feed["feed_version"],
    "snapshot_sha256": download_metadata["sha256"],
    "raw_snapshot_path": raw_snapshot_path,
}

try:
    for key, value in task_values.items():
        dbutils.jobs.taskValues.set(key=key, value=value)
    print("Task values publicados para tareas downstream.")
except Exception as error:
    print(f"Ejecución interactiva: task values no publicados ({error}).")

In [0]:
# BLOQUE 5 — Verificación de la capa Raw
#Bloque 5
#→ verificamos visualmente que el ZIP y el manifest existen.

print("Archivo GTFS estático:")
display(dbutils.fs.ls(snapshot_dir))

print("Manifest de esta corrida:")
display(dbutils.fs.ls(manifest_dir))

print("Contenido del manifest:")
print(dbutils.fs.head(manifest_path))